IMPORT LIBRARIES

In [ ]:
#import library untuk dataframe dan visualisasi 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

#import library untuk clustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans

LOAD DATA

In [ ]:
# Load data
df = pd.read_csv("data/transactions.csv")
df

In [ ]:
df['menu'].value_counts()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df['menu'].unique()

In [ ]:
# date formatting
df['date'] = pd.to_datetime(df['date'], format="%Y-%m-%d")
df['date']

In [ ]:
# Find the last date in the DataFrame
last_date = df['date'].max()

# Calculate three months before the last date
three_months_before = last_date - pd.DateOffset(months=3)

# Filter the DataFrame for dates three months before the last date
df = df[df['date'] >= three_months_before]
df

RFM-P

In [ ]:
# amount
df['Amount'] = df['revenue']
rfm_m = df.groupby('menu')['Amount'].sum()
rfm_m = rfm_m.reset_index() 
rfm_m

In [ ]:
# frequency
rfm_f = df.groupby('menu')['invoice'].count().reset_index()
rfm_f.columns = ['menu','Frequency']
rfm_f.head()

In [ ]:
# recency
df['date'] = pd.to_datetime(df['date'])
max_date = max(df['date'])
max_date

In [ ]:
df['Diff'] = max_date - df['date']
df.head()

In [ ]:
# Mengkalkulasi recency
rfm_p = df.groupby('menu')['Diff'].min()
rfm_p = rfm_p.reset_index()
rfm_p.head()

In [ ]:
rfm_p['Diff'] = rfm_p['Diff'].dt.days
rfm_p.head()

In [ ]:
# Menggabungkan kolom recency dengan frequency dan amount
rfm = pd.merge(rfm_m, rfm_p, on='menu', how='inner')
rfm.columns = ['menu', 'Amount', 'Recency']
rfm.head()

In [ ]:
# Menggabungkan kolom recency dengan frequency dan amount
rfm = pd.merge(rfm, rfm_f, on='menu', how='inner')
rfm.columns = ['menu', 'Amount', 'Recency', 'Frequency']
rfm.head()

In [ ]:
# Profit
profit = df.groupby('menu')['profit'].sum()
profit = profit.reset_index()
profit.head()

In [ ]:
rfm = pd.merge(rfm, profit, on='menu', how='inner')
rfm.columns = ['Menu', 'Amount', 'Recency', 'Frequency', 'Profit']
rfm.head()

In [ ]:
rfm.info()

In [ ]:
# handling outlier
# Analisis outlier untuk kolom amount, frequency, dan recency
pd.DataFrame.iteritems = pd.DataFrame.items
attributes = ['Amount', 'Frequency', 'Recency', 'Profit']
plt.rcParams['figure.figsize'] = [10,8]
sns.boxplot(data = rfm[attributes], orient="v", palette="Set2" ,whis=1.5,saturation=1, width=0.7)
plt.title("Outliers Variable Distribution", fontsize = 14, fontweight = 'bold')
plt.ylabel("Range", fontweight = 'bold')
plt.xlabel("Attributes", fontweight = 'bold')

In [ ]:
# Rescaling atribut/variabel
rfm_df = rfm[['Amount', 'Frequency', 'Recency', 'Profit']]

# Inisiasi StandardScaler
scaler = StandardScaler()

# Implementasi StandardScaler
rfm_df_scaled = scaler.fit_transform(rfm_df)
rfm_df_scaled.shape

In [ ]:
# Hasil StandardScaler
df_scaled = pd.DataFrame(rfm_df_scaled)
df_scaled.columns = ['Amount', 'Frequency', 'Recency','Profit']
df_scaled.head()

CLUSTERING

In [ ]:
# Elbow-curve/SSD
ssd = []
range_n_clusters = [2, 3, 4, 5, 6, 7, 8]
for num_clusters in range_n_clusters:
    kmeans = KMeans(n_clusters=num_clusters, max_iter=50, random_state=42)
    kmeans.fit(rfm_df_scaled)
    
    ssd.append(kmeans.inertia_)
    
# plot the SSDs for each n_clusters
plt.figure(figsize=(8, 6))

plt.plot(
    range_n_clusters,
    ssd,
    marker='o',
    linewidth=2
)

plt.axvline(
    x=3,
    linestyle='--',
    linewidth=1.5
)

plt.xlabel("Number of Clusters (K)", fontsize=16)
plt.ylabel("Within-Cluster Sum of Squares (WCSS)", fontsize=16)

plt.xticks(range_n_clusters, fontsize=13)
plt.yticks(fontsize=13)

plt.tight_layout()

plt.show()

In [ ]:
range_n_clusters = [2, 3, 4, 5, 6, 7, 8]

results = []

for num_clusters in range_n_clusters:

    kmeans = KMeans(
        n_clusters=num_clusters,
        max_iter=50,
        random_state=42,
    )

    cluster_labels = kmeans.fit_predict(rfm_df_scaled)

    silhouette = silhouette_score(
        rfm_df_scaled,
        cluster_labels
    )

    davies_bouldin = davies_bouldin_score(
        rfm_df_scaled,
        cluster_labels
    )

    results.append({
        "K": num_clusters,
        "Silhouette Score": silhouette,
        "Davies-Bouldin Index": davies_bouldin,
        "WCSS": kmeans.inertia_
    })

results_df = pd.DataFrame(results)

results_df

In [ ]:
# Final model with k=3
kmeans = KMeans(n_clusters=3, max_iter=50, random_state=42)
kmeans.fit(rfm_df_scaled)

In [ ]:
# K-Means label 
kmeans.labels_

In [ ]:
# assign the label
rfm['Cluster_Id'] = kmeans.labels_
rfm

VISUALISASI

In [ ]:
# Ukuran dan font
sns.set_context("paper", font_scale=1.4)

plt.figure(figsize=(8, 6), dpi=300)

sns.boxplot(
    x="Cluster_Id",
    y="Amount",
    data=rfm
)

# Label dan ukuran font
plt.xlabel("Cluster", fontsize=18)
plt.ylabel("Amount", fontsize=18)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Layout agar tidak terpotong
plt.tight_layout()

plt.show()

In [ ]:
# Ukuran dan font
sns.set_context("paper", font_scale=1.4)

plt.figure(figsize=(8, 6), dpi=300)

sns.boxplot(
    x="Cluster_Id",
    y="Frequency",
    data=rfm
)

# Label dan ukuran font
plt.xlabel("Cluster", fontsize=18)
plt.ylabel("Frequency", fontsize=18)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Layout agar tidak terpotong
plt.tight_layout()

plt.show()

In [ ]:
# Ukuran dan font
sns.set_context("paper", font_scale=1.4)

plt.figure(figsize=(8, 6), dpi=300)

sns.boxplot(
    x="Cluster_Id",
    y="Profit",
    data=rfm
)

# Label dan ukuran font
plt.xlabel("Cluster", fontsize=18)
plt.ylabel("Profit", fontsize=18)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Layout agar tidak terpotong
plt.tight_layout()

plt.show()

In [ ]:
# Ukuran dan font
sns.set_context("paper", font_scale=1.4)

plt.figure(figsize=(8, 6), dpi=300)

sns.boxplot(
    x="Cluster_Id",
    y="Recency",
    data=rfm
)

# Label dan ukuran font
plt.xlabel("Cluster", fontsize=18)
plt.ylabel("Recency", fontsize=18)

plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Layout agar tidak terpotong
plt.tight_layout()

plt.show()

In [ ]:
rfm.loc[rfm['Cluster_Id'] == 0, 'Cluster_Id'] = 'Silver' 
rfm.loc[rfm['Cluster_Id'] == 1, 'Cluster_Id'] = 'Gold'
rfm.loc[rfm['Cluster_Id'] == 2, 'Cluster_Id'] = 'Bronze' 
rfm.sort_values(by='Cluster_Id')

In [ ]:
rfm.info()

In [ ]:
rfm.to_csv('kmeans.csv', index=False)